In [1]:
#Load Libraries
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)
library(AnnotationHub)
library(GenomicRanges)
library(BiocParallel)
library(chromVAR)
library(JASPAR2024)
library(TFBSTools)
library(cicero)
library(monocle3)
library(SeuratWrappers)
library(ggplot2)
library(patchwork)
library(reticulate)
library(sceasy)
library(future)
library(Matrix)

#Set Options
options(future.globals.maxSize = 400000 * 1024^2) #for 300GB max size
plan("multicore", workers = 4)

register(SerialParam()) 
set.seed(1234)

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays please run: options(Seurat.object.assay.version = 'v5')

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position,

ERROR: Error in library(JASPAR2024): there is no package called 'JASPAR2024'


# Load Data

In [5]:
# load ATAC data
data.atac <- readRDS("checkpoints/DT_ATACintegrated.rds")
Idents(data.atac) <- 'dataset'
data.atac

An object of class Seurat 
569068 features across 210324 samples within 1 assay 
Active assay: ATAC (569068 features, 0 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: integrated_lsi, umap.integrated.atac

# Make Cicero object

In [4]:
ah <- AnnotationHub()

# Search for the Ensembl 98 EnsDb for Homo sapiens on AnnotationHub
query(ah, "EnsDb.Hsapiens.v98")
ensdb_v98 <- ah[["AH75011"]]

# extract gene annotations from EnsDb
annotation <- GetGRangesFromEnsDb(ensdb = ensdb_v98)

# change to UCSC style since the data was mapped to hg38
seqlevels(annotation) <- paste0('chr', seqlevels(annotation))
genome(annotation) <- "hg38"

rm(ah,ensdb_v98)

AnnotationHub with 1 record
# snapshotDate(): 2023-04-25
# names(): AH75011
# $dataprovider: Ensembl
# $species: Homo sapiens
# $rdataclass: EnsDb
# $rdatadateadded: 2019-05-02
# $title: Ensembl 98 EnsDb for Homo sapiens
# $description: Gene and protein annotations for Homo sapiens based on Ensem...
# $taxonomyid: 9606
# $genome: GRCh38
# $sourcetype: ensembl
# $sourceurl: http://www.ensembl.org
# $sourcesize: NA
# $tags: c("98", "AHEnsDbs", "Annotation", "EnsDb", "Ensembl", "Gene",
#   "Protein", "Transcript") 
# retrieve record with 'object[["AH75011"]]' 

downloading 1 resources

retrieving 1 resource

loading from cache

Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels i

In [6]:
# 1) Convert Seurat -> CDS
data.cds <- as.cell_data_set(data.atac)

# 2) Grab the UMAP you want from Seurat
umap <- Embeddings(data.atac, "umap.integrated.atac")  # rows = cells, cols = UMAP1/2

# 3) Make sure cell names match and are in the same order
common <- intersect(rownames(umap), colnames(data.cds))
data.cds <- data.cds[, common]          # subset cds to shared cells (keeps order of 'common')
umap  <- umap[common, , drop = FALSE]

# 4) Set it as the UMAP in the CDS (name must be exactly "UMAP")
reducedDims(data.cds)$UMAP <- umap

Warning message:
"`invoke()` is deprecated as of rlang 0.4.0.
Please use `exec()` or `inject()` instead.
This warning is displayed once every 8 hours."


In [7]:
# make cicero-cds object
cicero.cds <- make_cicero_cds(data.cds, reduced_coordinates = umap)

Overlap QC metrics:
Cells per bin: 50
Maximum shared cells bin-bin: 44
Mean shared cells bin-bin: 0.0114540287249344
Median shared cells bin-bin: 0



In [8]:
save_monocle_objects(cds=cicero.cds, directory_path='checkpoints/cicero')

  cds_object.rds  (full_cds  RDS  from  cell_data_set)



# Find Cicero connections

In [9]:
# get the chromosome sizes from the Seurat object
genome <- seqlengths(annotation)

# convert chromosome sizes to a dataframe
genome.df <- data.frame("chr" = names(genome), "length" = genome)

In [10]:
# run cicero
conns <- run_cicero(cicero.cds, genomic_coords = genome.df, silent = FALSE, sample_num = 100)

[1] "Starting Cicero"
[1] "Calculating distance_parameter value"
[1] "Running models"
[1] "Assembling connections"
[1] "Successful cicero models:  11512"
[1] "Other models: "

  Too many elements in range Zero or one element in range 
                           2                          853 
[1] "Models with errors:  0"
[1] "Done"


In [11]:
# Save as RDS
saveRDS(conns, file="checkpoints/cicero/connections.rds")

# Save Results

In [2]:
conns <- readRDS("checkpoints/cicero/connections.rds")
cicero.cds <- load_monocle_objects(directory_path='checkpoints/cicero')

In [3]:
# Save peaks for CellOracle
all_peaks <- row.names(exprs(cicero.cds))
all_peaks <- gsub('-','_',all_peaks)
write.csv(x = all_peaks, file = "checkpoints/cicero/all_peaks.csv")

conns$Peak1 <- gsub('-','_',conns$Peak1)
conns$Peak2 <- gsub('-','_',conns$Peak2)
write.csv(x = conns, file = "checkpoints/cicero/conns.csv")

In [4]:
sink("Notebooks/5_CellOracle/01_CiceroConnections-sessionInfo.txt")
sessionInfo()
sink()

R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/atlas/libblas.so.3.10.3 
LAPACK: /usr/lib/x86_64-linux-gnu/atlas/liblapack.so.3.10.3;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] grid      stats4    stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] Matrix_1.7-4                      future_1.67.0                    
 [3] sceasy_0.0.7                      reticulate_1.43.0                
 [5] patchwork_1.3.2                   ggplot2_4.0.0                    
 [7] SeuratWrap